# Case Study: Identificar MCAR, MAR e MNAR com MissingFCUP (Breast Cancer)

Este notebook demonstra, com visualizações simples, como reconhecer **MCAR**, **MAR** e **MNAR** usando o `MissingData`. 
A ideia é ser didático para pessoas que não vivem estatística no dia-a-dia (ex.: médicos, gestores, analistas).

**Objetivo prático**:
- Partir de um dataset completo (ground truth).
- Injetar 30% de missingness em três colunas com mecanismos diferentes.
- Usar gráficos do package para “ver” os padrões e explicar o que eles sugerem.

## 1. Setup

In [ ]:
# Se estiveres num ambiente limpo, podes instalar o package localmente:
# %pip install -q -e .

import numpy as np
import pandas as pd

from missingfcup.core.MissingData import MissingData
from missingfcup.plots.Panel import Panel

## 2. Dataset (Breast Cancer do scikit-learn)

Escolhi um dataset médico clássico e fácil de explicar. 
Antes de simular a ausência de dados, removemos possíveis NaNs para ter um **ground truth** completo.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)

df = data.frame.copy()

# Garantir que partimos de um dataset completo
# (neste caso, já vem completo, mas deixamos explícito)
df = df.dropna().reset_index(drop=True)

print(df.shape)
df.head()

## 3. Funções para simular MCAR, MAR e MNAR

Estas funções **injetam ~30% de missingness** em colunas específicas:

- **MCAR**: faltas completamente aleatórias.
- **MAR**: faltas dependem de outra variável observada.
- **MNAR**: faltas dependem do próprio valor (mais perigoso).

In [ ]:
def inject_mcar(df: pd.DataFrame, column: str, missing_rate: float = 0.30, seed: int = 42):
    # MCAR: remove valores aleatoriamente, sem padrão visível
    rng = np.random.default_rng(seed)
    n = len(df)
    n_missing = int(n * missing_rate)
    idx = rng.choice(n, size=n_missing, replace=False)
    df.loc[idx, column] = np.nan
    return idx


def inject_mar(
    df: pd.DataFrame,
    target_column: str,
    driver_column: str,
    missing_rate: float = 0.30,
    seed: int = 42,
    noise: float = 0.01,
):
    # MAR: probabilidade de falta depende de outra coluna observada
    rng = np.random.default_rng(seed)

    driver = df[driver_column].to_numpy()
    driver_scaled = (driver - driver.min()) / (driver.max() - driver.min() + 1e-9)

    # Score maior = maior probabilidade de falta
    score = driver_scaled + rng.normal(0.0, noise, size=len(df))
    cutoff = np.quantile(score, 1 - missing_rate)
    mask = score >= cutoff

    df.loc[mask, target_column] = np.nan
    return mask


def inject_mnar(
    df: pd.DataFrame,
    target_column: str,
    missing_rate: float = 0.30,
    seed: int = 42,
    noise: float = 0.01,
):
    # MNAR: probabilidade de falta depende do próprio valor da coluna
    rng = np.random.default_rng(seed)

    values = df[target_column].to_numpy()
    values_scaled = (values - values.min()) / (values.max() - values.min() + 1e-9)

    score = values_scaled + rng.normal(0.0, noise, size=len(df))
    cutoff = np.quantile(score, 1 - missing_rate)
    mask = score >= cutoff

    df.loc[mask, target_column] = np.nan
    return mask

## 4. Criar a versão com missingness

Vamos escolher três variáveis contínuas:
- **MCAR**: `mean texture`
- **MAR**: `mean smoothness` (depende de `mean radius`)
- **MNAR**: `mean area` (depende do próprio valor)

In [ ]:
# Definição das colunas
COL_MCAR = "mean texture"
COL_MAR = "mean smoothness"
COL_MNAR = "mean area"
DRIVER_MAR = "mean radius"

# Criar um dataset com missingness
_df_missing = df.copy()

inject_mcar(_df_missing, COL_MCAR, missing_rate=0.30, seed=7)
inject_mar(_df_missing, COL_MAR, DRIVER_MAR, missing_rate=0.30, seed=8)
inject_mnar(_df_missing, COL_MNAR, missing_rate=0.30, seed=9)

md = MissingData(_df_missing)

# Checar rapidamente a taxa de missingness nas 3 colunas-alvo
md.column_missing_rate.loc[[COL_MCAR, COL_MAR, COL_MNAR]].round(3)

## 5. Visão Geral

Aqui o objetivo é **dar a fotografia geral** do problema: 
quantas faltas existem e em que colunas.

In [ ]:
panel_overview = Panel(
    plots=[
        md.missing_count_barchart(
            title="Missing count por coluna",
            missing_color="#d62728",
            present_color="#2ca02c",
        ),
        md.column_missing_rate_heatmap(
            title="Missing rate por coluna",
            colorscale=[[0.0, "#2ca02c"], [1.0, "#d62728"]],
        ),
        md.heatmap(
            title="Mapa geral de missingness",
            show_colorscale=True,
            missing_color="#d62728",
            present_color="#2ca02c",
        ),
    ],
    title="Visão Geral das Ausências",
    max_cols=2,
)

panel_overview.show()

## 6. MCAR — Missing Completely At Random

**Como interpretar**:
- Se os pontos vermelhos aparecem espalhados sem padrão, isso sugere **ausência ao acaso**.
- Também esperamos **pouca correlação** entre a coluna MCAR e outras faltas.

In [ ]:
panel_mcar = Panel(
    plots=[
        md.scatterplot(
            x=COL_MCAR,
            y=DRIVER_MAR,
            title="MCAR: missing em 'mean texture' parece aleatório",
            missing_color="#d62728",
            present_color="#2ca02c",
        ),
        md.missingness_correlation_heatmap(
            selected_columns=[COL_MCAR, COL_MAR, COL_MNAR],
            title="Correlação de missingness (MCAR tende a ~0)",
        ),
    ],
    title="MCAR em evidência",
    max_cols=2,
)

panel_mcar.show()

## 7. MAR — Missing At Random

**Ideia chave**: a falta acontece porque **outra variável observável** influencia a ausência.

Aqui, a coluna `mean smoothness` fica ausente com maior frequência quando `mean radius` é maior.

In [ ]:
md.heatmap(
    selected_columns=[DRIVER_MAR, COL_MAR, COL_MCAR, COL_MNAR],
    order_by=[{"axis": "rows", "column": DRIVER_MAR, "type": "numeric", "ascending": False}],
    title="MAR: missingness concentrada em raios maiores",
    show_colorscale=True,
    missing_color="#d62728",
    present_color="#2ca02c",
).show()

md.parallel_coordinates(
    selected_columns=[DRIVER_MAR, COL_MAR, "mean perimeter", "mean texture"],
    missingness_color_column=COL_MAR,
    normalize=False,
    impute_below_range_frac=0.1,
    show_colorbar=True,
    title="MAR: linhas vermelhas aparecem mais em valores altos de radius",
    missing_color="#d62728",
    present_color="#2ca02c",
).show()

## 8. MNAR — Missing Not At Random

**Atenção**: este é o caso mais perigoso. 
Aqui a falta depende do **próprio valor da variável** (que já não vemos).

Ainda assim, um **proxy** (variável correlacionada) pode dar pistas.

In [ ]:
md.scatterplot(
    x=DRIVER_MAR,
    y=COL_MNAR,
    title="MNAR: missing em 'mean area' concentrado em valores altos de radius",
    missing_color="#d62728",
    present_color="#2ca02c",
).show()

md.upset_plot(
    selected_columns=[COL_MCAR, COL_MAR, COL_MNAR],
    title="Interseções de missingness (MCAR/MAR/MNAR)",
    missing_color="#d62728",
).show()

## 9. Padrões combinados (Missingness Patterns)

Este gráfico ajuda a ver **combinações de colunas** que faltam ao mesmo tempo.
É útil para perceber se o problema é isolado ou sistémico.

In [ ]:
md.pattern_barchart(
    selected_columns=[COL_MCAR, COL_MAR, COL_MNAR],
    value="percent",
    title="Padrões de missingness (3 colunas)",
    missing_color="#d62728",
).show()

## 10. Apêndice (opcional): métricas rápidas do `MissingData`

Para quem quiser confirmar números e padrões com dados tabulares.

In [ ]:
# Propriedades gerais
summary = {
    "rows": md.number_of_rows,
    "columns": md.number_of_columns,
    "total_missing_rate": round(md.total_missing_rate, 3),
    "total_missing_count": md.total_missing_count,
    "columns_complete": len(md.columns_complete),
    "rows_complete": len(md.rows_complete),
    "rows_with_missing": len(md.rows_with_missing),
}
summary

In [ ]:
# Máscaras e padrões (amostra)
md.missing_mask.head()

In [ ]:
md.observed_mask[:5]

In [ ]:
# Métricas por coluna
md.column_missing_rate.loc[[COL_MCAR, COL_MAR, COL_MNAR]].round(3)

In [ ]:
md.column_missing_count.loc[[COL_MCAR, COL_MAR, COL_MNAR]]

In [ ]:
md.column_present_count.loc[[COL_MCAR, COL_MAR, COL_MNAR]]

In [ ]:
md.column_missing_percent.loc[[COL_MCAR, COL_MAR, COL_MNAR]].round(1)

In [ ]:
md.column_completeness.loc[[COL_MCAR, COL_MAR, COL_MNAR]].round(2)

In [ ]:
# Métricas por linha (top 5 mais incompletas)
md.row_missing_rate.sort_values(ascending=False).head()

In [ ]:
md.row_missing_count.sort_values(ascending=False).head()

In [ ]:
md.row_missing_percent.sort_values(ascending=False).head().round(1)

In [ ]:
md.row_completeness.sort_values(ascending=True).head().round(2)

In [ ]:
# Padrões de missingness
md.missing_pattern_in_rows.head()

In [ ]:
len(md.missing_pattern_in_rows_unique)

In [ ]:
md.missing_pattern_counts(max_patterns=5)

In [ ]:
# Correlação de missingness (amostra)
md.missing_correlation().loc[[COL_MCAR, COL_MAR, COL_MNAR], [COL_MCAR, COL_MAR, COL_MNAR]].round(2)

In [ ]:
md.perfectly_correlated_missing_columns()

In [ ]:
md.rows_above_missing_threshold(0.2)[:10]

In [ ]:
md.columns_above_missing_threshold(0.2)